# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

Simple words, honest numbers. Everything below is **observed** on one sampled month (March 2026) and is **directional**, for decision-support only. This is a practical, **non-production** playbook: a weekly triage list for a human, not an automatic system.

**What this notebook turns into an action playbook:** the Week-4 rule (CTR below the norm for the page's position) and the Week-5/6 model (grouped-by-client, honest split). It adds reason codes, an archetype-to-action map, a decay/refresh insight, cost/value thinking, human-review rules, a no-go list, monitoring triggers, and exports for next week's paper.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### How a page gets its action
Every page that the rule can see (at least 100 impressions in days 1-21, average position 1-20) gets **exactly one reason code**, checked in this order. Data-quality gates come first so noisy pages are never pushed into action:

| Order | Reason code | Archetype in plain words | Action | Status |
|---|---|---|---|---|
| 1 | `THIN_EVIDENCE` | too few impressions or active days to trust | `HOLD_COLLECT_MORE_DATA` | HOLD |
| 2 | `SPIKE_DRIVEN` | one day supplies most of the impressions | `HOLD_VERIFY_SPIKE` | HOLD |
| 3 | `FADING_IMPRESSIONS` | impressions are drying up in the last observed week | `REFRESH_CONTENT` | ACT |
| 4 | `PAGE1_CTR_BELOW_NORM` | on page 1, but fewer clicks than typical for its position | `REWRITE_TITLE_META` | ACT |
| 5 | `STRIKING_DISTANCE_CTR_GAP` | positions 11-20 with a CTR gap | `IMPROVE_RELEVANCE_AND_LINKS` | ACT |
| 6 | `AT_OR_ABOVE_NORM` | nothing stands out | `MONITOR_ONLY` | MONITOR |

These archetypes are **rules, not clusters**. They are chosen for readability, so an editor can see why a page is in the list. They echo the paper's own advice (refresh before decay, refine page-one snippets, support positions 11-20) but are tested only against this month of data.

### How the queue is ranked
- **Value** = estimated clicks at stake over the next 30 days (an upper bound, not a forecast). For CTR-gap actions it is the Week-4 missed-clicks score scaled to 30 days. For `FADING_IMPRESSIONS` it is the larger of the CTR-gap value and the clicks lost per 30 days if the last-week drop in clicks stays. It is the larger of the two, never the sum.
- **Ranking rule:** the code compares "rule alone" and "rule x model persistence probability" on out-of-fold pages (grouped by client) and keeps the combined one only if it is not worse on both AUC and the top-5% review queue. Otherwise it falls back to the plain rule.

In [1]:
import os, json
import numpy as np
import pandas as pd
from huggingface_hub import login

# Token comes from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Add it in Colab -> Secrets and enable notebook access.")

os.environ["HF_TOKEN"] = token
login(token=token)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
!pip install -q huggingface_hub pandas pyarrow scikit-learn matplotlib

In [3]:
storage_options = {"token": token}
BASE = "hf://datasets/FlyRank/internship-warehouse"

df_raw = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options=storage_options)
df_content = pd.read_parquet(f"{BASE}/dim_content.parquet", storage_options=storage_options)
print("Performance rows:", len(df_raw), "| content rows:", len(df_content))

Performance rows: 9841378 | content rows: 519606


In [4]:

KEY = ["client_hash_id", "content_hash_id"]
need = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
        "gsc_sum_position", "client_has_gsc", "gsc_data_available"]
missing_cols = [c for c in need if c not in df_raw.columns]
assert not missing_cols, f"Missing columns in fact table: {missing_cols} -> rename them in this cell"

df = df_raw[(df_raw["client_has_gsc"] == True) & (df_raw["gsc_data_available"] == True)].copy()
df["report_date"] = pd.to_datetime(df["report_date"])
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    df[c] = df[c].fillna(0)
df["active"] = (df["gsc_impressions"] > 0).astype(int)

START = df["report_date"].min()
CUTOFF = START + pd.Timedelta(days=20)          # features: first 21 days
obs = df[df["report_date"] <= CUTOFF].copy()
later = df[df["report_date"] > CUTOFF].copy()   # label only, never a feature
print("Feature window:", obs["report_date"].min().date(), "to", obs["report_date"].max().date())
print("Label window:  ", later["report_date"].min().date(), "to", later["report_date"].max().date())

def agg_pages(d):
    return d.groupby(KEY).agg(
        impressions=("gsc_impressions", "sum"), clicks=("gsc_clicks", "sum"),
        sum_pos=("gsc_sum_position", "sum"), days_active=("active", "sum"),
        max_day_impr=("gsc_impressions", "max")).reset_index()

pages = agg_pages(obs)
pages = pages[pages["impressions"] > 0].copy()
pages["ctr"] = pages["clicks"] / pages["impressions"]
raw_pos = pages["sum_pos"] / pages["impressions"]
POSITION_OFFSET = 1 if raw_pos.min() < 1 else 0
pages["avg_position"] = raw_pos + POSITION_OFFSET
pages["spike_share"] = pages["max_day_impr"] / pages["impressions"]

MIN_IMPR, MAX_POS, MIN_POS = 100, 20, 1
RULE_BINS = [0, 1.5, 2.5, 3.5, 5, 7, 10, 15, 20]
RULE_LABELS = ["<=1.5", "1.5-2.5", "2.5-3.5", "3.5-5", "5-7", "7-10", "10-15", "15-20"]
pages["eligible"] = (pages["impressions"] >= MIN_IMPR) & (pages["avg_position"] >= MIN_POS) & (pages["avg_position"] <= MAX_POS)
pages["rule_bin"] = pd.cut(pages["avg_position"], bins=RULE_BINS, labels=RULE_LABELS)
norm = (pages[pages["eligible"]].groupby("rule_bin", observed=True)
        .agg(impressions=("impressions", "sum"), clicks=("clicks", "sum")).reset_index())
norm["expected_ctr"] = norm["clicks"] / norm["impressions"]
pages = pages.merge(norm[["rule_bin", "expected_ctr"]], on="rule_bin", how="left")

# ---- Features (first 21 days + static content only), same as Week 5 ----
d = obs.copy()
d["pos_day"] = d["gsc_sum_position"] / d["gsc_impressions"].where(d["gsc_impressions"] > 0) + POSITION_OFFSET
pos_std = d.groupby(KEY)["pos_day"].std().rename("pos_std").reset_index()
l7 = d[d["report_date"] > CUTOFF - pd.Timedelta(days=7)]
l7 = l7.groupby(KEY).agg(impr_l7=("gsc_impressions", "sum"), clicks_l7=("gsc_clicks", "sum")).reset_index()

M = pages[pages["eligible"]].copy()
M = M.merge(pos_std, on=KEY, how="left").merge(l7, on=KEY, how="left")
M["pos_std"] = M["pos_std"].fillna(0)
M[["impr_l7", "clicks_l7"]] = M[["impr_l7", "clicks_l7"]].fillna(0)
M["impr_l7_share"] = M["impr_l7"] / M["impressions"]
M["ctr_l7"] = np.where(M["impr_l7"] > 0, M["clicks_l7"] / M["impr_l7"].where(M["impr_l7"] > 0), M["ctr"])

cont = df_content[["content_hash_id", "search_volume", "word_count", "char_count", "competition_level"]].drop_duplicates("content_hash_id")
M = M.merge(cont, on="content_hash_id", how="left")
for c in ["search_volume", "word_count", "char_count"]:
    M[f"{c}_missing"] = M[c].isna().astype(int)
M["comp_missing"] = M["competition_level"].isna().astype(int)
M["competition"] = M["competition_level"].astype(str).str.upper().map({"LOW": 1, "MEDIUM": 2, "HIGH": 3}).fillna(0)
M["log_impr"] = np.log1p(M["impressions"])
M["log_vol"] = np.log1p(M["search_volume"].fillna(0))
M["log_words"] = np.log1p(M["word_count"].fillna(0))
M["log_chars"] = np.log1p(M["char_count"].fillna(0))

FEATURES = ["log_impr", "ctr", "avg_position", "days_active", "spike_share", "pos_std",
            "impr_l7_share", "ctr_l7", "log_vol", "log_words", "log_chars", "competition",
            "search_volume_missing", "word_count_missing", "char_count_missing", "comp_missing"]

lp = agg_pages(later)[KEY + ["impressions", "clicks"]].rename(columns={"impressions": "later_impr", "clicks": "later_clicks"})
M = M.merge(lp, on=KEY, how="left")
M[["later_impr", "later_clicks"]] = M[["later_impr", "later_clicks"]].fillna(0)
M = M.reset_index(drop=True)
print("Modeling pages:", len(M), "| clients:", M["client_hash_id"].nunique(), "| features:", len(FEATURES))

Feature window: 2026-03-01 to 2026-03-21
Label window:   2026-03-22 to 2026-03-31
Modeling pages: 68868 | clients: 38 | features: 16


In [5]:
# ---- Honest persistence probability for every page: out-of-fold, grouped by client ----
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

M["rule_missed"] = np.clip(M["expected_ctr"] * M["impressions"] - M["clicks"], 0, None)          # Week-4 score
M["missed_pooled"] = np.clip(M["expected_ctr"] * M["later_impr"] - M["later_clicks"], 0, None)   # later window (check only)

n_clients = M["client_hash_id"].nunique()
n_splits = min(5, n_clients)
assert n_splits >= 3, f"Only {n_clients} clients: grouped validation needs at least 3"

M["persist_prob"] = np.nan
M["y_oof"] = np.nan
for fold, (tr, te) in enumerate(GroupKFold(n_splits=n_splits).split(M, groups=M["client_hash_id"])):
    thr = max(float(M["missed_pooled"].iloc[tr].quantile(0.75)), 1.0)      # threshold from TRAIN pages only
    y = (M["missed_pooled"] >= thr).astype(int)
    if y.iloc[tr].nunique() < 2:
        continue
    mdl = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=42)
    mdl.fit(M.iloc[tr][FEATURES], y.iloc[tr])
    M.loc[M.index[te], "persist_prob"] = mdl.predict_proba(M.iloc[te][FEATURES])[:, 1]
    M.loc[M.index[te], "y_oof"] = y.iloc[te].to_numpy()

print("Pages with an out-of-fold probability:", int(M["persist_prob"].notna().sum()), "of", len(M))
M["persist_prob"] = M["persist_prob"].fillna(M["persist_prob"].median())

# ---- Which ranking rule do we trust? Evaluate on out-of-fold pages ----
ok = M["y_oof"].notna().to_numpy()
E = M[ok]
k = max(10, int(np.ceil(0.05 * len(E))))
cands = {"rule": E["rule_missed"], "model": E["persist_prob"], "combined (rule x probability)": E["rule_missed"] * E["persist_prob"]}
rows = []
for nm, sc in cands.items():
    top = np.argsort(-sc.to_numpy(), kind="stable")[:k]
    rows.append({"ranking": nm, "AUC": roc_auc_score(E["y_oof"], sc),
                 "captured_later_missed_top5%": E["missed_pooled"].to_numpy()[top].sum() / max(E["missed_pooled"].sum(), 1e-9),
                 "precision_top5%": E["y_oof"].to_numpy()[top].mean()})
rank_eval = pd.DataFrame(rows).set_index("ranking")
print("RANKING CANDIDATES (out-of-fold, grouped by client; top 5% = the review queue)")
print(rank_eval.round(3).to_string())

r, c = rank_eval.loc["rule"], rank_eval.loc["combined (rule x probability)"]
RANKING_CHOICE = "combined" if (c["AUC"] >= r["AUC"] and c["captured_later_missed_top5%"] >= r["captured_later_missed_top5%"]) else "rule"
print(f"\nRANKING CHOICE: {RANKING_CHOICE}")
print("Note: chosen on the same pages it is scored on, so any edge is slightly optimistic. Directional only.")

Pages with an out-of-fold probability: 68868 of 68868
RANKING CANDIDATES (out-of-fold, grouped by client; top 5% = the review queue)
                                 AUC  captured_later_missed_top5%  precision_top5%
ranking                                                                           
rule                           0.797                        0.413            0.946
model                          0.913                        0.393            0.952
combined (rule x probability)  0.820                        0.414            0.950

RANKING CHOICE: combined
Note: chosen on the same pages it is scored on, so any edge is slightly optimistic. Directional only.


In [6]:
# ---- Archetypes, reason codes, actions, values ----
THIN_IMPR, THIN_DAYS, SPIKE_SHARE, FADE_SHARE = 300, 10, 0.30, 0.20     # honest, editable thresholds

PLAYBOOK = {
    "THIN_EVIDENCE": dict(status="HOLD", action="HOLD_COLLECT_MORE_DATA", effort_h=0.0,
        trigger=f"impressions < {THIN_IMPR} or fewer than {THIN_DAYS} active days in days 1-21",
        human_check="Do nothing yet. Re-score after another window; check the page is not new or just indexed.",
        wrong_if="the page is new and healthy but has not had time to collect data",
        measure="Re-score at the next window"),
    "SPIKE_DRIVEN": dict(status="HOLD", action="HOLD_VERIFY_SPIKE", effort_h=0.25,
        trigger=f"one day supplies at least {SPIKE_SHARE:.0%} of impressions",
        human_check="Open Search Console for the page: was it a one-off event, a news pickup or a tracking glitch?",
        wrong_if="the spike is a real, lasting gain in demand",
        measure="Check whether the following weeks still show the demand"),
    "FADING_IMPRESSIONS": dict(status="ACT", action="REFRESH_CONTENT", effort_h=3.0,
        trigger=f"less than {FADE_SHARE:.0%} of the 21-day impressions fall in the last 7 days (steady would be about 33%)",
        human_check="Read the page. Is it dated, thin or overtaken by a competitor? Refresh only pages worth keeping; check the query is not seasonal.",
        wrong_if="the drop is seasonal or the topic itself lost demand, so a refresh cannot help",
        measure="Impressions, clicks and average position at 30 and 60 days versus untouched similar pages"),
    "PAGE1_CTR_BELOW_NORM": dict(status="ACT", action="REWRITE_TITLE_META", effort_h=0.5,
        trigger="average position 10 or better and CTR below the norm for that position band",
        human_check="Look at the live results page. Is the query branded or navigational? Does a snippet, answer box or AI answer take the clicks? Only then draft a tighter title and description.",
        wrong_if="the query is branded or a SERP feature explains the low CTR, so a rewrite cannot win clicks back",
        measure="Weighted CTR and clicks for revised pages over the next 30 days versus comparable unrevised pages"),
    "STRIKING_DISTANCE_CTR_GAP": dict(status="ACT", action="IMPROVE_RELEVANCE_AND_LINKS", effort_h=1.5,
        trigger="average position 10-20 and CTR below the norm for that band",
        human_check="Check intent match for the target query, add internal links from stronger related pages, tighten the opening answer.",
        wrong_if="the page ranks for many unrelated queries, so the average position hides what is happening",
        measure="Movement into positions 1-10 and click change over 30-60 days"),
    "AT_OR_ABOVE_NORM": dict(status="MONITOR", action="MONITOR_ONLY", effort_h=0.0,
        trigger="CTR at or above the norm and no other warning sign",
        human_check="No action. Keep in the monthly re-score.",
        wrong_if="a slow decline is hiding behind a good CTR",
        measure="Monthly re-score"),
}

M["clicks_first14"] = (M["clicks"] - M["clicks_l7"]).clip(lower=0)
M["decay_clicks_30d"] = np.clip(M["clicks_first14"] / 14 - M["clicks_l7"] / 7, 0, None) * 30
M["decay_impr_30d"] = np.clip((M["impressions"] - M["impr_l7"]) / 14 - M["impr_l7"] / 7, 0, None) * 30

conds = [
    (M["impressions"] < THIN_IMPR) | (M["days_active"] < THIN_DAYS),
    M["spike_share"] >= SPIKE_SHARE,
    M["impr_l7_share"] < FADE_SHARE,
    (M["rule_missed"] > 0) & (M["avg_position"] <= 10),
    (M["rule_missed"] > 0),
]
codes = ["THIN_EVIDENCE", "SPIKE_DRIVEN", "FADING_IMPRESSIONS", "PAGE1_CTR_BELOW_NORM", "STRIKING_DISTANCE_CTR_GAP"]
M["reason_code"] = np.select(conds, codes, default="AT_OR_ABOVE_NORM")
M["action"] = M["reason_code"].map({k: v["action"] for k, v in PLAYBOOK.items()})
M["status"] = M["reason_code"].map({k: v["status"] for k, v in PLAYBOOK.items()})
M["effort_hours_assumed"] = M["reason_code"].map({k: v["effort_h"] for k, v in PLAYBOOK.items()})

gap_value = M["rule_missed"] * 30 / 21
M["value_clicks_30d"] = np.where(M["reason_code"] == "FADING_IMPRESSIONS", np.maximum(M["decay_clicks_30d"], gap_value),
                         np.where(M["status"] == "ACT", gap_value, 0.0))

# Ranking: model probability only scales CTR-gap actions (it was trained on the CTR-gap label, not on decay)
use_prob = (RANKING_CHOICE == "combined") & M["reason_code"].isin(["PAGE1_CTR_BELOW_NORM", "STRIKING_DISTANCE_CTR_GAP"])
M["rank_score"] = M["value_clicks_30d"] * np.where(use_prob, M["persist_prob"], 1.0)

def confidence(r):
    if r.impressions >= 1000 and r.days_active >= 14 and r.spike_share < 0.30:
        return "HIGH"
    if r.impressions >= 300 and r.days_active >= 7:
        return "MEDIUM"
    return "LOW"
M["confidence"] = M.apply(confidence, axis=1)

order = {"ACT": 0, "HOLD": 1, "MONITOR": 2}
M["_o"] = M["status"].map(order)
queue = M.sort_values(["_o", "rank_score", "impressions"], ascending=[True, False, False]).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue["human_review_required"] = True
queue["safe_to_automate"] = False
queue["review_note"] = queue["reason_code"].map({k: v["human_check"] for k, v in PLAYBOOK.items()})

mix = M.groupby(["status", "reason_code"]).agg(pages=("impressions", "size"), impressions=("impressions", "sum"),
        value_clicks_30d=("value_clicks_30d", "sum")).reset_index()
print("QUEUE MIX (pages, impressions and estimated clicks at stake per 30 days)")
print(mix.round(1).to_string(index=False))
print("\nTOP 15 ACTIONS (hashed ids shortened; no client names or URLs)")
view = queue[queue["status"] == "ACT"].head(15).copy()
view["page"] = view["content_hash_id"].astype(str).str[-6:]
print(view[["rank", "page", "reason_code", "action", "value_clicks_30d", "persist_prob", "confidence",
            "impressions", "avg_position", "ctr", "expected_ctr"]].round(3).to_string(index=False))

QUEUE MIX (pages, impressions and estimated clicks at stake per 30 days)
 status               reason_code  pages  impressions  value_clicks_30d
    ACT        FADING_IMPRESSIONS   5031     10161081           44645.5
    ACT      PAGE1_CTR_BELOW_NORM  20838     62625369          165330.6
    ACT STRIKING_DISTANCE_CTR_GAP   7291     11062296           30552.9
   HOLD              SPIKE_DRIVEN    798      2040250               0.0
   HOLD             THIN_EVIDENCE  19060      4098571               0.0
MONITOR          AT_OR_ABOVE_NORM  15850     45997133               0.0

TOP 15 ACTIONS (hashed ids shortened; no client names or URLs)
 rank   page               reason_code                      action  value_clicks_30d  persist_prob confidence  impressions  avg_position   ctr  expected_ctr
    1 a5a1b3        FADING_IMPRESSIONS             REFRESH_CONTENT           537.857         0.019       HIGH        15209         6.345 0.024         0.003
    2 ae744a        FADING_IMPRESSIONS       

### Decay / refresh insight

The paper's refresh advice is observational. The check below asks a smaller question that **my own data can answer**: when a page's impressions are already fading in the last observed week, do they keep falling in the next 10 days, hold at the lower level, or rebound? That decides whether `REFRESH_CONTENT` is aimed at a persistent problem or at noise.

The comparison uses daily impression rates: first 14 days, last 7 days of the feature window, and the 10-day later window (check only, never a feature).

In [7]:
M["momentum"] = pd.cut(M["impr_l7_share"], [-0.01, FADE_SHARE, 0.50, 10], labels=["fading", "steady", "surging"])
g = M.groupby("momentum", observed=True).agg(pages=("impressions", "size"), first14=("impressions", "sum"),
        last7=("impr_l7", "sum"), later10=("later_impr", "sum")).reset_index()
g["first14_per_day"] = (g["first14"] - g["last7"]) / 14 / g["pages"]
g["last7_per_day"] = g["last7"] / 7 / g["pages"]
g["later_per_day"] = g["later10"] / 10 / g["pages"]
g["later_vs_last7"] = g["later_per_day"] / g["last7_per_day"]
decay = g[["momentum", "pages", "first14_per_day", "last7_per_day", "later_per_day", "later_vs_last7"]]
print("DAILY IMPRESSIONS PER PAGE by momentum group (n printed)")
print(decay.round(2).to_string(index=False))

f = decay.set_index("momentum")
if "fading" in f.index:
    ratio = float(f.loc["fading", "later_vs_last7"])
    if ratio < 0.90:
        DECAY_READ = "keep falling"
    elif ratio <= 1.10:
        DECAY_READ = "hold at the lower level"
    else:
        DECAY_READ = "rebound"
    print(f"\nFading pages: next-10-day rate is {ratio:.2f}x their last-7-day rate, so they {DECAY_READ} (n = {int(f.loc['fading','pages'])} pages).")
    print("Caveats: weekday mix differs between 7 and 10 days, and pages picked for a low last week tend to regress toward their average.")
    DECAY_N = int(f.loc["fading", "pages"])
else:
    ratio, DECAY_READ, DECAY_N = float("nan"), "no fading pages found", 0
    print("No fading pages in this sample.")

DAILY IMPRESSIONS PER PAGE by momentum group (n printed)
momentum  pages  first14_per_day  last7_per_day  later_per_day  later_vs_last7
  fading   8131            85.80          28.57          44.46            1.56
  steady  50681           105.91         107.25         124.18            1.16
 surging  10056            29.36         103.15         130.88            1.27

Fading pages: next-10-day rate is 1.56x their last-7-day rate, so they rebound (n = 8131 pages).
Caveats: weekday mix differs between 7 and 10 days, and pages picked for a low last week tend to regress toward their average.


## 2. Intended use and limits

*Who uses this, for what, and where it stops being valid.*

### Intended use
- **Who:** a content editor or SEO strategist who has about an hour a week for triage.
- **For what:** an **ordered list of pages worth a human look first**, each with one reason code, one suggested action and a note on what to check. It is decision-support for choosing what to review, not a verdict on any page.
- **Cadence:** re-score monthly (see Section 4), review the ACT list from the top, and log the outcome of each review.
- **Form:** a CSV and a short table. Nothing publishes, edits or redirects automatically.

### Where it stops being valid
- **One month, one sample:** observed on March 2026 only, GSC-available pages only, pages with at least 100 impressions and average position 1-20 in days 1-21.
- **Proxy value:** "clicks at stake" is the gap between a page's CTR and the pooled norm for its position band. It is an upper bound, not a forecast, and it is not a measured result of any rewrite.
- **Page-level, not query-level:** branded queries, answer boxes and AI answers can make a low CTR normal.
- **Average position hides swings:** a page that is #1 on some days and #15 on others looks like position 8.
- **Pooled norm is in-sample:** it describes what is typical this month, not what is achievable.
- **Model edge is modest:** the model helps the order mainly through AUC; on the top-5% queue it did not clearly beat the plain rule in Weeks 5-6, which is why the code checks before using it.
- **No causal claim:** nothing here shows that refreshing, rewriting or linking recovers clicks. A small before/after pilot would be needed for that.

In [8]:
# ---- Coverage: how much of the portfolio does the playbook actually speak for? ----
all_pages = agg_pages(obs)
all_pages = all_pages[all_pages["impressions"] > 0]
cov_pages = len(M) / len(all_pages)
cov_impr = M["impressions"].sum() / all_pages["impressions"].sum()
print(f"Eligible pages: {len(M):,} of {len(all_pages):,} pages with impressions ({cov_pages:.1%}); they hold {cov_impr:.1%} of all impressions in days 1-21.")
print("Clients covered:", M["client_hash_id"].nunique())
print("\nShare of eligible pages by status:")
print((M["status"].value_counts(normalize=True) * 100).round(1).to_string())

# ---- Cost / value thinking (all effort numbers are ASSUMPTIONS to be replaced by the editorial team) ----
REVIEW_CAPACITY_PER_WEEK = 25            # assumption: pages one reviewer can act on per week
act = queue[queue["status"] == "ACT"].copy()
act["priority_per_hour"] = np.where(act["effort_hours_assumed"] > 0, act["value_clicks_30d"] / act["effort_hours_assumed"], 0.0)

by_action = act.groupby("action").agg(pages=("impressions", "size"), value_clicks_30d=("value_clicks_30d", "sum"),
        effort_h_per_page=("effort_hours_assumed", "first")).reset_index()
by_action["clicks_at_stake_per_effort_hour"] = by_action["value_clicks_30d"] / (by_action["pages"] * by_action["effort_h_per_page"])
print("\nVALUE vs EFFORT by action (effort hours are assumptions; value is an upper bound)")
print(by_action.round(2).to_string(index=False))

total_value = float(act["value_clicks_30d"].sum())
cum = act["value_clicks_30d"].cumsum() / max(total_value, 1e-9)
n_week = min(REVIEW_CAPACITY_PER_WEEK, len(act))
share_week = float(cum.iloc[n_week - 1]) if len(act) else 0.0
n_half = int((cum < 0.5).sum() + 1) if len(act) else 0
print(f"\nTotal estimated clicks at stake across {len(act):,} ACT pages: {total_value:,.0f} per 30 days (upper bound).")
print(f"One reviewing week ({n_week} pages) touches {share_week:.1%} of that value; {n_half} pages hold half of it.")
print(f"Weeks to clear the whole ACT list at that pace: {len(act) / REVIEW_CAPACITY_PER_WEEK:.1f}")

Eligible pages: 68,868 of 162,301 pages with impressions (42.4%); they hold 75.9% of all impressions in days 1-21.
Clients covered: 38

Share of eligible pages by status:
status
ACT        48.2
HOLD       28.8
MONITOR    23.0

VALUE vs EFFORT by action (effort hours are assumptions; value is an upper bound)
                     action  pages  value_clicks_30d  effort_h_per_page  clicks_at_stake_per_effort_hour
IMPROVE_RELEVANCE_AND_LINKS   7291          30552.88                1.5                             2.79
            REFRESH_CONTENT   5031          44645.54                3.0                             2.96
         REWRITE_TITLE_META  20838         165330.64                0.5                            15.87

Total estimated clicks at stake across 33,160 ACT pages: 240,529 per 30 days (upper bound).
One reviewing week (25 pages) touches 2.7% of that value; 3367 pages hold half of it.
Weeks to clear the whole ACT list at that pace: 1326.4


### Cost / value in plain words
- The value is concentrated: a short list at the top holds a large share of the estimated clicks at stake, so a small weekly capacity is worth spending in rank order.
- A title and description rewrite is cheap (assumed 0.5 hours), a content refresh is expensive (assumed 3 hours), so clicks at stake **per effort hour** can reorder what feels most urgent. The effort numbers are placeholders for the editorial team to replace.
- Because value is an upper bound and the recovery rate is unknown, do not turn these numbers into a revenue promise. Treat them as a way to order the work.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-review rules (every row, no exceptions)
1. **A person reads the page and the live results page** before any change. The queue column `review_note` says what to look for per reason code.
2. **Check query intent first.** Branded or navigational queries, answer boxes and AI answers can explain a low CTR; do not rewrite those.
3. **Check the data is trustworthy.** HOLD pages (thin evidence, one-day spikes) are not worked until a person has verified them.
4. **One change at a time** per page where possible, so the follow-up can say what moved.
5. **Log the outcome** of every review (acted, rejected and why, no change) so the monitoring triggers in Section 4 have something to read.
6. **A second person approves** anything that changes public content.

### The no-go list: never automate these
- **Never auto-publish** a rewritten title, description or page. Drafts are fine; publishing is a human act.
- **Never auto-delete, redirect, merge or de-index** a page because of its score.
- **Never act on HOLD pages** without verification, and never auto-clear a HOLD.
- **Never use the score to judge a person, a writer or a client.** It ranks pages for review, not performance.
- **Never promise a click gain** to a client or in a report from these numbers.
- **Never describe this as predicting Google or "the algorithm".**
- **Never run it on pages or clients without Search Console data**, or on data older than the monitoring window allows.
- **Never put client names, URLs or private queries** in the queue, a figure or a paper.

The exported queue carries `human_review_required = True` and `safe_to_automate = False` on every row so the rule travels with the data.

In [9]:
# ---- The review checklist as data (this table is exported for the paper) ----
checklist = pd.DataFrame([
    {"reason_code": k, "status": v["status"], "action": v["action"], "trigger": v["trigger"],
     "human_check": v["human_check"], "wrong_if": v["wrong_if"], "measure_after": v["measure"],
     "effort_hours_assumed": v["effort_h"]} for k, v in PLAYBOOK.items()])
pd.set_option("display.max_colwidth", 70)
print(checklist[["reason_code", "action", "human_check"]].to_string(index=False))

NO_GO = [
    "auto-publish rewritten titles, descriptions or pages",
    "auto-delete, redirect, merge or de-index pages from a score",
    "act on HOLD pages without verification",
    "judge a person, writer or client with the score",
    "promise click gains to a client from these numbers",
    "describe the work as predicting Google or the algorithm",
    "run on pages or clients without Search Console data",
    "put client names, URLs or private queries in any output",
]
assert (queue["human_review_required"].all()) and (not queue["safe_to_automate"].any())
print("\nGuard passed: every queue row requires human review and none is marked safe to automate.")

              reason_code                      action                                                                                                                                                                    human_check
            THIN_EVIDENCE      HOLD_COLLECT_MORE_DATA                                                                                      Do nothing yet. Re-score after another window; check the page is not new or just indexed.
             SPIKE_DRIVEN           HOLD_VERIFY_SPIKE                                                                                  Open Search Console for the page: was it a one-off event, a news pickup or a tracking glitch?
       FADING_IMPRESSIONS             REFRESH_CONTENT                                              Read the page. Is it dated, thin or overtaken by a competitor? Refresh only pages worth keeping; check the query is not seasonal.
     PAGE1_CTR_BELOW_NORM          REWRITE_TITLE_META Look at the live results page.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a light, monthly routine, not a production monitor. The automated checks below compare **equal-length windows** so that window length does not fake a change. The demo here compares the first 10 days of March with the last 10 days; in real use, compare consecutive months.

| Trigger | What it watches | Threshold (editable) | If it fires |
|---|---|---|---|
| Norm drift | CTR for each position band | any band (at least 30 pages) moved by more than 25% | Re-learn the norm before trusting the ranking |
| Spike share | share of pages that are `SPIKE_DRIVEN` | moved by more than 10 percentage points | Check tracking and data definitions |
| Model gain | model-over-rule AUC under grouped CV | falls below 0.02 | Retire the model ranking and use the plain rule |
| Reviewer rejection | share of reviewed ACT pages rejected (branded, SERP feature) | above 30% | Revise the rule or the reason-code order |
| Data definition | GSC columns, position convention, missing share | any change | Re-run Weeks 3-6 checks before use |
| Time | calendar | monthly re-score, quarterly full re-validation | Re-run this notebook on the newest month |

In [10]:
def window_stats(d, days):
    p = agg_pages(d)
    p = p[p["impressions"] > 0].copy()
    p["avg_position"] = p["sum_pos"] / p["impressions"] + POSITION_OFFSET
    p["ctr"] = p["clicks"] / p["impressions"]
    p["spike_share"] = p["max_day_impr"] / p["impressions"]
    min_impr = MIN_IMPR * days / 21                                   # scale the eligibility bar to the window length
    e = p[(p["impressions"] >= min_impr) & (p["avg_position"] >= MIN_POS) & (p["avg_position"] <= MAX_POS)].copy()
    e["bin"] = pd.cut(e["avg_position"], bins=RULE_BINS, labels=RULE_LABELS)
    b = e.groupby("bin", observed=True).agg(n=("ctr", "size"), imp=("impressions", "sum"), clk=("clicks", "sum"))
    return {"norm": (b["clk"] / b["imp"]), "n_by_bin": b["n"], "spike_share_pages": float((e["spike_share"] >= SPIKE_SHARE).mean()), "n_pages": int(len(e))}

def check_triggers(ref, cur, model_gain_auc):
    out = []
    shared = ref["norm"].index.intersection(cur["norm"].index)
    shared = [b for b in shared if ref["n_by_bin"][b] >= 30 and cur["n_by_bin"][b] >= 30 and ref["norm"][b] > 0]
    rel = {b: abs(cur["norm"][b] - ref["norm"][b]) / ref["norm"][b] for b in shared}
    worst = max(rel, key=rel.get) if rel else None
    out.append(("Norm drift", f"largest band change {rel[worst]:.0%} (band {worst})" if worst else "no band with enough pages",
                "FIRE" if worst and rel[worst] > 0.25 else "ok"))
    ds = abs(cur["spike_share_pages"] - ref["spike_share_pages"])
    out.append(("Spike share", f"{ref['spike_share_pages']:.1%} -> {cur['spike_share_pages']:.1%}", "FIRE" if ds > 0.10 else "ok"))
    out.append(("Model gain", f"AUC gain over rule {model_gain_auc:+.3f}", "FIRE" if model_gain_auc < 0.02 else "ok"))
    return pd.DataFrame(out, columns=["trigger", "reading", "status"])

first10 = df[df["report_date"] <= START + pd.Timedelta(days=9)]
last10 = df[df["report_date"] > df["report_date"].max() - pd.Timedelta(days=10)]
ref_stats, cur_stats = window_stats(first10, 10), window_stats(last10, 10)
model_gain = float(rank_eval.loc["model", "AUC"] - rank_eval.loc["rule", "AUC"])
trig = check_triggers(ref_stats, cur_stats, model_gain)
print("TRIGGER DEMO: first 10 days (reference) vs last 10 days (current) of March")
print(trig.to_string(index=False))
print("\nReference norm (CTR by position band, first 10 days):")
print(ref_stats["norm"].round(4).to_string())
print("\nThis is a demonstration of the check, not a real drift test. Real use: compare consecutive months.")

TRIGGER DEMO: first 10 days (reference) vs last 10 days (current) of March
    trigger                              reading status
 Norm drift largest band change 65% (band <=1.5)   FIRE
Spike share                         7.8% -> 5.8%     ok
 Model gain            AUC gain over rule +0.116     ok

Reference norm (CTR by position band, first 10 days):
bin
<=1.5      0.0007
1.5-2.5    0.0028
2.5-3.5    0.0046
3.5-5      0.0040
5-7        0.0030
7-10       0.0027
10-15      0.0036
15-20      0.0033

This is a demonstration of the check, not a real drift test. Real use: compare consecutive months.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/`. The paper builds on these files.*

- `work/outputs/action_queue.csv`: the ranked queue. It stays out of git by design (the leak-guard blocks data files); this notebook regenerates it.
- `work/figures/*.png`: figures to **commit** and reuse.
- `work/outputs/action_playbook.json` and `work/outputs/playbook_metrics.json`: the playbook table, assumptions, thresholds and numbers, with no data rows. **Commit these**; they are the receipts the paper's numbers trace back to.

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# ---- Queue CSV (hashed ids only; stays out of git) ----
cols = ["rank", "client_hash_id", "content_hash_id", "status", "reason_code", "action", "value_clicks_30d", "rank_score",
        "persist_prob", "rule_missed", "confidence", "impressions", "clicks", "ctr", "expected_ctr", "avg_position",
        "days_active", "spike_share", "impr_l7_share", "effort_hours_assumed", "human_review_required", "safe_to_automate", "review_note"]
queue[cols].to_csv("work/outputs/action_queue.csv", index=False)
print("Wrote work/outputs/action_queue.csv | rows:", len(queue), "| ACT:", int((queue["status"] == "ACT").sum()))

# ---- Figure 1: value concentration ----
fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(1, len(act) + 1)
ax.plot(x, cum.to_numpy() * 100, color="#1f4e79")
ax.axvline(n_week, color="#b45f06", linestyle="--", linewidth=1)
ax.set_xlabel("Pages reviewed in rank order (ACT queue)")
ax.set_ylabel("Share of estimated clicks at stake (%)")
ax.set_title("Value concentration in the ACT queue (observed, March 2026)")
ax.text(n_week, 5, f" one week = {n_week} pages", color="#b45f06", fontsize=9)
ax.set_xscale("log")
fig.tight_layout(); fig.savefig("work/figures/fig_value_concentration.png", dpi=160); plt.close(fig)

# ---- Figure 2: action mix ----
am = mix[mix["status"] == "ACT"].sort_values("value_clicks_30d", ascending=False)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.barh(am["reason_code"], am["value_clicks_30d"], color="#1f4e79")
ax.invert_yaxis()
ax.set_xlabel("Estimated clicks at stake per 30 days (upper bound)")
ax.set_title("Where the value sits, by reason code")
fig.tight_layout(); fig.savefig("work/figures/fig_action_mix.png", dpi=160); plt.close(fig)

# ---- Figure 3: decay / momentum ----
if len(decay):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    w = 0.25; idx = np.arange(len(decay))
    ax.bar(idx - w, decay["first14_per_day"], w, label="first 14 days", color="#9fc5e8")
    ax.bar(idx, decay["last7_per_day"], w, label="last 7 days", color="#3d85c6")
    ax.bar(idx + w, decay["later_per_day"], w, label="next 10 days (check)", color="#0b5394")
    ax.set_xticks(idx); ax.set_xticklabels([f"{m}\n(n={n})" for m, n in zip(decay["momentum"], decay["pages"])])
    ax.set_ylabel("Impressions per page per day")
    ax.set_title("Do fading pages keep fading? (observed)")
    ax.legend()
    fig.tight_layout(); fig.savefig("work/figures/fig_decay_momentum.png", dpi=160); plt.close(fig)
print("Wrote figures:", sorted(os.listdir("work/figures")))

# ---- Playbook table + receipts (no data rows; safe to commit) ----
playbook_out = {
    "name": "Content action playbook (non-production)", "month": "2026-03",
    "order_of_checks": list(PLAYBOOK.keys()),
    "archetypes": checklist.to_dict(orient="records"),
    "no_go_list": NO_GO,
    "thresholds": {"min_impressions": MIN_IMPR, "position_range": [MIN_POS, MAX_POS], "thin_impressions": THIN_IMPR,
                   "thin_days": THIN_DAYS, "spike_share": SPIKE_SHARE, "fade_share": FADE_SHARE},
    "assumptions": {"review_capacity_per_week": REVIEW_CAPACITY_PER_WEEK, "effort_hours_are_assumptions": True,
                    "value_is_upper_bound_not_forecast": True},
    "monitoring_triggers": {"norm_drift_rel": 0.25, "spike_share_points": 0.10, "model_auc_gain_min": 0.02,
                            "reviewer_rejection_max": 0.30, "cadence": "monthly re-score, quarterly full validation"},
}
metrics_out = {
    "assignment": "ML-10 action playbook",
    "split_for_probabilities": f"GroupKFold by client, {n_splits} folds, {n_clients} clients",
    "eligible_pages": int(len(M)), "coverage_pages_share": round(float(cov_pages), 4), "coverage_impressions_share": round(float(cov_impr), 4),
    "status_counts": M["status"].value_counts().to_dict(), "reason_code_counts": M["reason_code"].value_counts().to_dict(),
    "ranking_evaluation": rank_eval.round(4).to_dict(orient="index"), "ranking_choice": RANKING_CHOICE,
    "total_value_clicks_30d_upper_bound": round(total_value, 1), "pages_for_half_of_value": n_half,
    "one_week_share_of_value": round(share_week, 4),
    "decay_insight": {"fading_pages": DECAY_N, "later_vs_last7_ratio": (None if np.isnan(ratio) else round(float(ratio), 3)), "reading": DECAY_READ},
    "trigger_demo": trig.to_dict(orient="records"),
}
with open("work/outputs/action_playbook.json", "w") as f:
    json.dump(playbook_out, f, indent=2, default=str)
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2, default=str)
print("Wrote work/outputs/action_playbook.json and work/outputs/playbook_metrics.json")

Wrote work/outputs/action_queue.csv | rows: 68868 | ACT: 33160
Wrote figures: ['fig_action_mix.png', 'fig_decay_momentum.png', 'fig_value_concentration.png']
Wrote work/outputs/action_playbook.json and work/outputs/playbook_metrics.json


### Safe sentences the paper can reuse
The cell below writes recommendation sentences from the numbers above, so the paper cannot drift from this notebook.

In [12]:
n_act, n_hold = int((M["status"] == "ACT").sum()), int((M["status"] == "HOLD").sum())
s1 = (f"In one sampled month (March 2026), the playbook covered {len(M):,} pages ({cov_impr:.0%} of impressions in the first 21 days) and "
      f"placed {n_act:,} in an ACT list and {n_hold:,} on HOLD for human verification.")
s2 = (f"Estimated clicks at stake in the ACT list are an upper bound of about {total_value:,.0f} per 30 days; "
      f"{n_half:,} pages hold half of it, so a short, ranked weekly review is where the observed value concentrates.")
s3 = (f"Among pages already fading in the last observed week (n = {DECAY_N:,}), their impressions in the following 10 days "
      f"{DECAY_READ} (observed ratio {ratio:.2f} against the last 7 days)" +
      (", which is directional support for reviewing fading pages, not proof that a refresh recovers them." if DECAY_READ != "rebound"
       else ", so many fading weeks reverse on their own and a person should verify a fading page before refreshing it."))
s4 = ("These are decision-support rankings for what a person should review first. Every action needs human review; "
      "none is safe to automate, and no click gain is promised.")
for s in (s1, s2, s3, s4):
    print("-", s)

- In one sampled month (March 2026), the playbook covered 68,868 pages (76% of impressions in the first 21 days) and placed 33,160 in an ACT list and 19,858 on HOLD for human verification.
- Estimated clicks at stake in the ACT list are an upper bound of about 240,529 per 30 days; 3,367 pages hold half of it, so a short, ranked weekly review is where the observed value concentrates.
- Among pages already fading in the last observed week (n = 8,131), their impressions in the following 10 days rebound (observed ratio 1.56 against the last 7 days), so many fading weeks reverse on their own and a person should verify a fading page before refreshing it.
- These are decision-support rankings for what a person should review first. Every action needs human review; none is safe to automate, and no click gain is promised.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all), with outputs saved
- [ ] No client names, URLs, tokens, or private queries anywhere (only hashed ids, and the CSV stays out of git)
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] I replaced the effort-hour and weekly-capacity assumptions if I know better numbers
- [ ] I wrote 2-3 sentences of my own on the top 15 actions (do they look sensible to a human?)
- [ ] Committed `work/notebooks/w07_action_playbook.ipynb`, `work/figures/*.png`, `work/outputs/action_playbook.json` and `work/outputs/playbook_metrics.json` (not the CSV), then submit the repo URL on the card. Done.